# Plot-Shift — Colab Scorer

Runs any `SentimentScorer` implementation on windows exported from the local pipeline.

## Round-trip
```
# 1. local
uv run plot-shift export-colab-data --out-csv colab_input.csv

# 2. Colab: upload colab_input.csv, run all cells, download colab_scores.csv

# 3. local
uv run plot-shift import-colab-scores --csv-path colab_scores.csv
```

## Swapping the scorer
Replace only the **Scorer implementation** cell with any class that inherits
`SentimentScorer` from the cell above it.  Everything else stays the same.

In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available.")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# ── Sentiment abstractions (copied verbatim from src/plot_shift/sentiment/base.py) ──
#
# Do NOT edit this cell.  It is the shared contract every scorer below implements.

from abc import ABC, abstractmethod
from dataclasses import dataclass
from enum import Enum


class ScorerKind(Enum):
    LEXICON     = "lexicon"
    TRANSFORMER = "transformer"


class DimensionKind(Enum):
    VALENCE   = "valence"
    AROUSAL   = "arousal"
    DOMINANCE = "dominance"


@dataclass(frozen=True, slots=True)
class DimensionScore:
    value: float
    kind:  DimensionKind


@dataclass()
class Score:
    def __init__(self) -> None:
        self._dimensions: list[DimensionScore] = []

    def add_dimension_score(self, dimension: DimensionScore) -> None:
        self._dimensions.append(dimension)

    @property
    def dimensions(self) -> list[DimensionScore]:
        return self._dimensions


class SentimentScorer(ABC):
    """Any model that maps a unit of text to a Score."""

    @abstractmethod
    def score(self, text: str) -> Score: ...

    @property
    @abstractmethod
    def scorer_name(self) -> str: ...

    @property
    @abstractmethod
    def scorer_kind(self) -> ScorerKind: ...

In [ ]:
!pip install -q transformers torch pandas huggingface_hub

In [ ]:
import transformers
import huggingface_hub

print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

In [ ]:
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from torch import nn
from transformers import AutoTokenizer, RobertaModel

from plot_shift.sentiment import DimensionScore, Score, SentimentScorer
from plot_shift.sentiment.base import DimensionKind, ScorerKind


class RoBERTaForVAD(nn.Module):
    """
    Architecture used by PERT-EmoPair.

    RoBERTa-large encoder + dropout + linear regression head
    producing Valence, Arousal and Dominance.
    """

    def __init__(self, base_model_name: str = "roberta-large"):
        super().__init__()

        self.base_model = RobertaModel.from_pretrained(base_model_name)

        self.dropout = nn.Dropout(p=0.1)

        self.regressor = nn.Linear(
            self.base_model.config.hidden_size,
            3,
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:

        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # First token = <s> / CLS equivalent for RoBERTa
        cls = self.dropout(outputs.last_hidden_state[:, 0, :])

        return self.regressor(cls)


class PERTEmoPairScorer(SentimentScorer):
    """
    PERT-EmoPair transformer scorer.

    Produces three continuous emotional dimensions:

        - Valence
        - Arousal
        - Dominance
    """

    REPO_ID = "edsi-umd/PERT-EmoPair"
    BASE_MODEL = "roberta-large"

    PROMPT = (
        "- Valence: The emotional valence of the context is <mask>. "
        "- Arousal: The level of arousal of the context is <mask>. "
        "- Dominance: The perceived dominance associated with the context is <mask>. "
        "Please predict the missing value for each dimension using the content provided."
    )

    def __init__(
        self,
        device: str | None = None,
        max_length: int = 256,
    ):
        self.max_length = max_length

        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"

        self.device = torch.device(device)

        print(f"Using device: {self.device}")

        # --------------------------------------------------
        # Tokenizer
        # --------------------------------------------------

        self.tokenizer = AutoTokenizer.from_pretrained(self.REPO_ID)

        # --------------------------------------------------
        # Model architecture
        # --------------------------------------------------

        self.model = RoBERTaForVAD(base_model_name=self.BASE_MODEL)

        # --------------------------------------------------
        # Download PERT-EmoPair checkpoint
        # --------------------------------------------------

        model_dir = snapshot_download(repo_id=self.REPO_ID)

        checkpoint_path = Path(model_dir) / "pytorch_model.bin"

        if not checkpoint_path.exists():
            raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

        # --------------------------------------------------
        # Load checkpoint
        # --------------------------------------------------

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True,
        )

        self.model.load_state_dict(checkpoint["model_state_dict"])

        # --------------------------------------------------
        # Evaluation mode
        # --------------------------------------------------

        self.model.to(self.device)
        self.model.eval()

        print("PERT-EmoPair loaded successfully.")

    def _build_prompt(self, text: str) -> str:
        return f"Context: '{text}' {self.PROMPT}"

    def _predict(self, text: str) -> list[float]:

        prompted_text = self._build_prompt(text)

        encoded = self.tokenizer(
            prompted_text,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
        )

        encoded = {key: value.to(self.device) for key, value in encoded.items()}

        with torch.inference_mode():
            output = self.model(**encoded)

        return output.squeeze(0).detach().cpu().tolist()

    def score(self, text: str) -> Score:

        if not text or not text.strip():
            raise ValueError("Text must not be empty.")

        valence, arousal, dominance = self._predict(text)

        result = Score()

        result.add_dimension_score(
            DimensionScore(
                value=float(valence),
                kind=DimensionKind.VALENCE,
            )
        )

        result.add_dimension_score(
            DimensionScore(
                value=float(arousal),
                kind=DimensionKind.AROUSAL,
            )
        )

        result.add_dimension_score(
            DimensionScore(
                value=float(dominance),
                kind=DimensionKind.DOMINANCE,
            )
        )

        return result

    @property
    def scorer_name(self) -> str:
        return "PERT-EmoPair"

    @property
    def scorer_kind(self) -> ScorerKind:
        return ScorerKind.TRANSFORMER


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────

INPUT_CSV   = "/content/colab_input.csv"          # exported by plot-shift export-colab-data
OUTPUT_CSV  = "/content/colab_scores.csv"          # import with plot-shift import-colab-scores
WEIGHTS_PATH = "/content/pert_emopar_weights.pt"  # upload your .pt file here

In [ ]:
# ── Load input CSV ─────────────────────────────────────────────────────────────

import pandas as pd
from pathlib import Path

required = {"imdb_id_int", "window_idx", "total_windows",
            "from_segment_idx", "to_segment_idx", "window_text"}

df = pd.read_csv(INPUT_CSV)
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Input CSV missing columns: {sorted(missing)}")

print(f"Loaded {len(df):,} windows from {INPUT_CSV}")
df.head()

In [ ]:
# ── Instantiate scorer ─────────────────────────────────────────────────────────
#
# Change this cell when you swap the scorer implementation above.

scorer: SentimentScorer = PERTEmoPairScorer()

print(f"Scorer : {scorer.scorer_name}")
print(f"Kind   : {scorer.scorer_kind.value}")

In [ ]:
# ── Score windows and write output CSV ────────────────────────────────────────
#
# One output row per (window, dimension).  All columns required by
# `plot-shift import-colab-scores` are written here.

passthrough = ["imdb_id_int", "window_idx", "total_windows",
               "from_segment_idx", "to_segment_idx"]

rows_out = []
for i, row in df.iterrows():
    window_score = scorer.score(str(row["window_text"]))
    for dim in window_score.dimensions:
        rows_out.append({
            **{col: row[col] for col in passthrough},
            "scorer_name": scorer.scorer_name,
            "scorer_kind": scorer.scorer_kind.value,
            "dimension":   dim.kind.value,
            "score":       dim.value,
        })

    if (i + 1) % 500 == 0:
        print(f"  {i + 1:,} / {len(df):,} windows scored…")

out_df = pd.DataFrame(rows_out)
out_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nWrote {len(out_df):,} rows → {OUTPUT_CSV}")
out_df.head()

In [ ]:
# ── Download from Colab ────────────────────────────────────────────────────────
from google.colab import files
files.download(OUTPUT_CSV)